# Post 009 — Time Series Forecasting
## Dataset B: Silicon Aging (BTI Degradation) Forecasting

**AI Engineering Lab Series | Era 1: Classic Machine Learning**

---

**Bias Temperature Instability (BTI)** is one of the primary aging mechanisms in modern CMOS transistors. Over time, repeated switching causes threshold voltage (Vth) to drift upward, slowing down the circuit. If Vth drifts too far, the chip fails its timing specifications.

This notebook applies time series forecasting to predict Vth degradation over a chip's operational lifetime, enabling engineers to predict **when** a chip will fail its timing spec — before it happens in the field.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded')

In [ ]:
df = pd.read_csv('../data/silicon_aging_bti.csv', parse_dates=['timestamp'])
print(f'Shape: {df.shape}')
print(f'Time range: {df["timestamp"].min()} to {df["timestamp"].max()}')
print(f'\nDevices: {df["device_id"].nunique()}')
print(f'\nVth stats:')
print(df['vth_mv'].describe())
df.head()

## 1. BTI Degradation Physics

BTI degradation follows a power-law relationship with time:

$$\Delta V_{th}(t) = A \cdot t^n$$

Where:
- $A$ = degradation coefficient (depends on voltage, temperature, process)
- $n$ ≈ 0.25 for NBTI in PMOS (the time exponent)
- $t$ = stress time

This means degradation is fastest early in life and slows down over time. The challenge: we measure only the first few thousand hours and need to predict behavior at 10+ years (87,600 hours).

In [ ]:
# Plot degradation curves for multiple devices
device_ids = df['device_id'].unique()[:8]
colors = plt.cm.tab10(np.linspace(0, 1, len(device_ids)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

for dev, col in zip(device_ids, colors):
    dev_data = df[df['device_id'] == dev].sort_values('hours')
    ax1.plot(dev_data['hours'], dev_data['vth_mv'], color=col, alpha=0.7, linewidth=1.5, label=dev)
    ax2.plot(np.log10(dev_data['hours'] + 1), dev_data['vth_mv'], color=col, alpha=0.7, linewidth=1.5)

# Add failure threshold
threshold = df['vth_mv'].mean() + 30  # 30mV drift = timing failure
ax1.axhline(y=threshold, color='red', linestyle='--', linewidth=2, label=f'Failure threshold ({threshold:.0f}mV)')
ax2.axhline(y=threshold, color='red', linestyle='--', linewidth=2)

ax1.set_title('BTI Degradation: Vth vs Linear Time')
ax1.set_xlabel('Stress Hours'); ax1.set_ylabel('Threshold Voltage (mV)')
ax1.legend(fontsize=7, loc='upper left')

ax2.set_title('BTI Degradation: Vth vs Log Time (Power-Law Linearized)')
ax2.set_xlabel('log10(Stress Hours)'); ax2.set_ylabel('Threshold Voltage (mV)')

plt.tight_layout()
plt.show()
print(f'Failure threshold: {threshold:.1f} mV (30 mV drift from initial)')

## 2. Single Device: ARIMA Forecasting

We take one representative device, train on the first 3,000 hours of measurements, and forecast the remaining 2,000 hours. The goal is to predict when Vth will cross the failure threshold.

In [ ]:
# Use device 0 as the primary example
dev_data = df[df['device_id'] == device_ids[0]].sort_values('hours').reset_index(drop=True)
vth_series = dev_data['vth_mv']

# Train/test split at 3000 hours
split_idx = (dev_data['hours'] <= 3000).sum()
train_vth = vth_series.iloc[:split_idx]
test_vth = vth_series.iloc[split_idx:]
train_hours = dev_data['hours'].iloc[:split_idx]
test_hours = dev_data['hours'].iloc[split_idx:]

print(f'Train: {len(train_vth)} measurements (0-3000 hrs)')
print(f'Test:  {len(test_vth)} measurements (3000-5000 hrs)')

# Stationarity test
result = adfuller(train_vth)
print(f'\nADF p-value: {result[1]:.4f} → {"Stationary" if result[1] < 0.05 else "Non-stationary, needs differencing"}')

In [ ]:
# Fit ARIMA
arima = ARIMA(train_vth, order=(2, 1, 1))
arima_fit = arima.fit()

# Forecast
forecast_result = arima_fit.get_forecast(steps=len(test_vth))
forecast_mean = forecast_result.predicted_mean
conf_int = forecast_result.conf_int(alpha=0.1)  # 90% CI

mae = mean_absolute_error(test_vth.values, forecast_mean.values)
rmse = np.sqrt(mean_squared_error(test_vth.values, forecast_mean.values))
print(f'ARIMA(2,1,1) — MAE: {mae:.3f} mV | RMSE: {rmse:.3f} mV')

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train_hours, train_vth, color='steelblue', label='Training data', linewidth=1.5)
ax.plot(test_hours, test_vth, color='black', label='Actual (held out)', linewidth=2)
ax.plot(test_hours, forecast_mean.values, color='coral', linestyle='--', label='ARIMA Forecast', linewidth=2)
ax.fill_between(test_hours, conf_int.iloc[:, 0], conf_int.iloc[:, 1], 
                color='coral', alpha=0.15, label='90% Confidence Interval')
ax.axhline(y=threshold, color='red', linestyle=':', linewidth=1.5, label=f'Failure threshold ({threshold:.0f}mV)')
ax.axvline(x=3000, color='gray', linestyle=':', linewidth=1, label='Forecast start')
ax.set_title('BTI Vth Degradation: ARIMA Forecast with Failure Threshold')
ax.set_xlabel('Stress Hours'); ax.set_ylabel('Threshold Voltage (mV)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Predicting Time-to-Failure

The most valuable output is not the Vth forecast itself, but the predicted **time-to-failure** — when will Vth cross the threshold? This directly informs field replacement schedules and warranty policies.

In [ ]:
# Extend forecast to find time-to-failure
extended_steps = 10000  # forecast far into the future
extended_forecast = arima_fit.forecast(steps=extended_steps)
extended_hours = np.arange(len(train_vth), len(train_vth) + extended_steps) * (dev_data['hours'].diff().median())

# Find first crossing of threshold
failure_indices = np.where(extended_forecast.values >= threshold)[0]
if len(failure_indices) > 0:
    failure_hour = extended_hours[failure_indices[0]]
    print(f'Predicted time-to-failure: {failure_hour:.0f} hours ({failure_hour/8760:.1f} years)')
else:
    print('No failure predicted within forecast horizon')

# Compare across all devices
print('\nTime-to-failure estimates across all devices:')
print(f'{"Device":<15} {"Predicted TTF (hrs)":<22} {"Predicted TTF (years)"}')
print('-' * 55)
for dev in device_ids:
    dev_d = df[df['device_id'] == dev].sort_values('hours')
    vth_s = dev_d['vth_mv'].values
    try:
        m = ARIMA(vth_s, order=(2, 1, 1)).fit()
        ext = m.forecast(steps=extended_steps)
        fi = np.where(ext.values >= threshold)[0]
        if len(fi) > 0:
            ttf = extended_hours[fi[0]]
            print(f'{dev:<15} {ttf:<22.0f} {ttf/8760:.1f}')
        else:
            print(f'{dev:<15} {"No failure":<22} -')
    except:
        print(f'{dev:<15} {"Model error":<22} -')

## 4. Summary

This notebook demonstrated how ARIMA-based time series forecasting can predict silicon aging and estimate time-to-failure for individual chips.

**Engineering impact**: Instead of running chips for 10,000 hours in accelerated life testing (ALT) to observe failure, we can now predict failure time from the first 3,000 hours of data — a 70% reduction in test time and cost.

**Key takeaways:**
1. **BTI degradation is predictable** — the power-law physics means the trajectory is smooth and forecastable
2. **ARIMA on log-transformed data** often works better for monotonically increasing degradation signals
3. **Confidence intervals are critical** — a narrow CI means we can commit to a warranty period; a wide CI means we need more data
4. **Time-to-failure is the real deliverable** — not the forecast curve itself, but the crossing point with the failure threshold